In [166]:
import torch
import torch.nn as nn
import torchvision.transforms as Transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import torchvision


In [167]:
my_transform = Transforms.Compose([
    Transforms.ToTensor(),
    Transforms.Normalize((0.5,),(0.5,))
])

In [168]:
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=my_transform,download=True)
test_dataset = torchvision.datasets.MNIST(root='./data',train= False, transform=my_transform,download=True)

In [169]:
train_dataloader = DataLoader(dataset= train_dataset,batch_size = 64, shuffle = True)
test_dataloader = DataLoader(dataset= test_dataset, batch_size= 64, shuffle = False)

In [170]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3,stride=1,padding=0)
        self.relu = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        self.conv2  = nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,stride=1,padding=0)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        self.fc = nn.Linear(in_features=32*5*5,out_features=10)

    def forward(self,x):

        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))

        x = x.reshape(x.size(0),-1)

        x = self.fc(x)

        return x    

In [171]:
class CNN_2(nn.Module):
    def __init__(self):
        super(CNN_2,self).__init__()

        #self.network = nn.Sequential()

        self.conv1 = nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3,stride=1,padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        
        self.conv2 = nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        
        self.conv3 = nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,stride=1,padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=1)


        self.fc = nn.Linear(in_features=64*7*7,out_features=10)

    def forward(self,x):

        x = self.pool1(self.relu1(self.conv1(x)))

        x = self.pool2(self.relu2(self.conv2(x)))

        x = self.pool3(self.relu3(self.conv3(x)))

        x = x.reshape(x.size(0),-1)

        x = self.fc(x)

        return x


In [172]:
model = CNN_2()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr = 0.025)

In [173]:
num_epochs = 25

for epochs in range(num_epochs):

    for i,(images,labels) in enumerate(train_dataloader):

        output = model(images)

        loss = criterion(output,labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        if (i+1)%100 == 0:
            print(f"epoch: {epochs+1}, step: {i+1}, loss: {loss.item()}")

epoch: 1, step: 100, loss: 1.0703301429748535
epoch: 1, step: 200, loss: 0.705255389213562
epoch: 1, step: 300, loss: 0.28414300084114075
epoch: 1, step: 400, loss: 0.2165040671825409
epoch: 1, step: 500, loss: 0.17714382708072662
epoch: 1, step: 600, loss: 0.21393422782421112
epoch: 1, step: 700, loss: 0.20870257914066315
epoch: 1, step: 800, loss: 0.049122393131256104
epoch: 1, step: 900, loss: 0.17346352338790894
epoch: 2, step: 100, loss: 0.11781608313322067
epoch: 2, step: 200, loss: 0.26463621854782104
epoch: 2, step: 300, loss: 0.14458249509334564
epoch: 2, step: 400, loss: 0.17929111421108246
epoch: 2, step: 500, loss: 0.043378796428442
epoch: 2, step: 600, loss: 0.0631549283862114
epoch: 2, step: 700, loss: 0.09127446264028549
epoch: 2, step: 800, loss: 0.06732236593961716
epoch: 2, step: 900, loss: 0.12419352680444717
epoch: 3, step: 100, loss: 0.11625898629426956
epoch: 3, step: 200, loss: 0.21529272198677063
epoch: 3, step: 300, loss: 0.01693497784435749
epoch: 3, step: 400

In [174]:
model.eval()

correct_count = 0
total_count = 0

with torch.no_grad():
    for images,labels in test_dataloader:

        output = model(images)

        _, prediction = torch.max(output.data,1)

        total_count += labels.size(0)
        correct_count += (prediction == labels).sum().item()



accuracy = 100 * correct_count / total_count
print(f'Accuracy on the 10,000 test images: {accuracy:.2f}%')

Accuracy on the 10,000 test images: 99.04%
